PYSPARK DATAFRAMES
│
├── 1. DataFrame Fundamentals
│   ├── What is a DataFrame?
│   ├── RDD vs DataFrame
│   ├── DataFrame architecture
│   ├── Catalyst Optimizer
│   └── Tungsten Engine
│
├── 2. Creating DataFrames
│   ├── From Python list
│   ├── From RDD
│   ├── Using schema
│   ├── StructType / StructField
│   └── Reading CSV / JSON / Parquet

        - spark.read.csv()
        - spark.read.json()
        - spark.read.parquet()
        
│
├── 3. Understanding DataFrames
│   ├── show()
│   ├── printSchema()
│   ├── schema
│   ├── columns
│   ├── dtypes
│   ├── count()
│   └── describe()
│
├── 4. Column Operations
│   ├── select()
│   ├── col()
│   ├── alias()
│   ├── withColumn()
│   ├── withColumnRenamed()
│   ├── drop()
│   └── cast()
│
├── 5. Filtering
│   ├── filter()
│   ├── where()
│   ├── AND / OR / NOT
│   ├── isin()
│   ├── between()
│   └── NULL handling
│
├── 6. Conditional Logic
│   ├── when()
│   ├── otherwise()
│   └── Multiple conditions
│
├── 7. Aggregations
│   ├── groupBy()
│   ├── count()
│   ├── sum()
│   ├── avg()
│   ├── min()
│   ├── max()
│   └── agg()
│
├── 8. Joins
│   ├── inner
│   ├── left
│   ├── right
│   ├── full
│   ├── left_semi
│   ├── left_anti
│   ├── cross
│   └── broadcast join
│
├── 9. Sorting & Duplicates
│   ├── orderBy()
│   ├── sort()
│   ├── distinct()
│   └── dropDuplicates()
│
├── 10. Null Handling
│   ├── isNull()
│   ├── isNotNull()
│   ├── dropna()
│   ├── fillna()
│   └── coalesce()
│
├── 11. String / Date Functions
│
├── 12. Window Functions
│   ├── row_number()
│   ├── rank()
│   ├── dense_rank()
│   ├── lag()
│   └── lead()
│
├── 13. explode / array / struct / map
│
├── 14. UDF vs Built-in Functions
│
├── 15. Repartition / Coalesce / Partitioning
│
├── 16. Cache / Persist
│
├── 17. DataFrame Execution Plan
│   ├── explain()
│   ├── Logical Plan
│   ├── Optimized Logical Plan
│   └── Physical Plan
│
├── 18. Performance Optimization
│   ├── Predicate pushdown
│   ├── Column pruning
│   ├── Broadcast joins
│   ├── Shuffle optimization
│   ├── Data skew
│   └── AQE
│
└── 19. Production Data Engineering Project
    ├── Raw ingestion
    ├── Schema validation
    ├── Data quality
    ├── Reject handling
    ├── Transformations
    ├── Aggregations
    ├── Joins
    ├── Partitioning
    └── Parquet output

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataFrame_Learning")
    .master("local[*]")
    .getOrCreate()
)

print("SparkSession created successfully")

In [ ]:
print("Spark Version:", spark.version)
print("Application Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default Parallelism:", spark.sparkContext.defaultParallelism)


In [ ]:
spark.stop()

What is a Spark DataFrame?

A Spark DataFrame is a distributed collection of data organized into named columns.

+-----------+----------+-------+
|customer_id|product   |amount |
+-----------+----------+-------+
|C101       |Laptop    |65000  |
|C102       |Mobile    |30000  |
|C103       |Keyboard  |2000   |
+-----------+----------+-------+

But unlike a normal Python table or Pandas DataFrame, the data can be distributed across multiple Spark partitions and multiple machines.

                     Spark DataFrame
                           |
          ---------------------------------
          |               |               |
      Partition 0     Partition 1     Partition 2
          |               |               |
       Records          Records          Records

[
    ("C101", "Laptop", 65000),
    ("C102", "Mobile", 30000),
    ("C103", "Keyboard", 2000),
    ("C104", "Monitor", 15000),
    ("C105", "Mouse", 1000),
    ("C106", "Laptop", 70000)
]


Partition 0
C101 Laptop    65000
C102 Mobile    30000

Partition 1
C103 Keyboard   2000
C104 Monitor   15000

Partition 2
C105 Mouse      1000
C106 Laptop    70000


So the DataFrame looks like one table to us, but Spark processes different portions of it in parallel.

A DataFrame also has a schema. 

root
 |-- customer_id: string
 |-- product: string
 |-- amount: long

 

## How a Spark Dataframe is Distributed

## Why Does Spark Use Partition?





In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataFrameLearning")
    .master("local[*]")
    .getOrCreate()
)

data = [
    ("C101", "Laptop", 65000),
    ("C102", "Mobile", 30000),
    ("C103", "Keyboard", 2000)
]

columns = ["customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

In [ ]:
df.printSchema()

In [ ]:
df.filter(df.amount > 10000).select(
    "customer_id",
    "product"
).show()

# Dataframe Defenition 

A Spark DataFrame is an immutable, distributed collection of data organized into named columns with an associated schema. It provides higher-level APIs than RDDs and allows Spark SQL’s Catalyst optimizer and execution engine to optimize query execution.


# RDD vs DataFrame

## Comparison

```text
+--------------------------+--------------------------------------+--------------------------------------+
| Feature                  | RDD                                  | DataFrame                            |
+--------------------------+--------------------------------------+--------------------------------------+
| Abstraction Level        | Low-level API                        | High-level API                       |
| Data Structure           | Distributed objects / records        | Distributed rows and columns         |
| Schema                   | No built-in schema                   | Has schema                           |
| Column Names             | Not directly available               | Available                            |
| Data Access              | Index / object based                 | Column-name based                    |
| Example                  | x[3]                                 | df.amount                            |
| Catalyst Optimizer       | Not available                        | Available                            |
| SQL Support              | No direct SQL support                | Supports Spark SQL                   |
| Query Optimization       | Limited                              | Strong optimization                  |
| Readability              | Lower for structured data            | Higher                               |
| Low-Level Control        | More                                 | Less than RDD                        |
| Transformations          | map(), flatMap(), reduceByKey()      | select(), filter(), groupBy(), join()|
| Lazy Evaluation          | Yes                                  | Yes                                  |
| Fault Tolerance          | Yes                                  | Yes                                  |
| Distributed Processing   | Yes                                  | Yes                                  |
| Best Use Case            | Custom low-level processing          | Structured Data Engineering          |
| Structured ETL           | Less preferred                       | Generally preferred                  |
| Performance              | Usually lower for structured ETL     | Usually better for structured ETL    |
+--------------------------+--------------------------------------+--------------------------------------+

In [ ]:
data=[
    ("C101", "Laptop", 65000),
    ("C102", "Mobile", 30000),
    ("C103", "Keyboard", 2000),
    ("C104", "Monitor", 15000),
    ("C105", "Mouse", 1000),
    ("C106", "Laptop", 70000),
    ("C101", "Laptop", 65000),
    ("C102", "Mobile", 30000),
    ("C103", "Keyboard", 2000),
    ("C104", "Monitor", 15000),
    ("C105", "Mouse", 1000),
    ("C106", "Laptop", 70000),
    ("C101", "Laptop", 65000),
    ("C102", "Mobile", 30000),
    ("C103", "Keyboard", 2000),
    ("C104", "Monitor", 15000),
    ("C105", "Mouse", 1000),
    ("C106", "Laptop", 70000)
]

columns=["customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

In [ ]:
df.rdd.getNumPartitions()

In [ ]:
df_4=df.repartition(4)

In [ ]:
df_4.rdd.getNumPartitions()

In [ ]:
from pyspark.sql import functions as F
df_partitioned=df_4.withColumn("partition_id", F.spark_partition_id())
df_partitioned.show(20, False)


## Dataframe Architecture 



In [ ]:
result_df=(
    df.filter(df.amount > 10000)
    .select("customer_id", "product")
)
result_df.show()

Pyspark Datframe Code  - You write Transformation like filter(), select(),groupBy(),join() 

Unresolved Logical Plan - Spark understand what operations you requested but columns/tables are not fully validated yet

Analyzed Logical Plan - Spark checks schema,column name,data type,tables,alaias,etc 

Optimized Logical Plan - Catalyst optimizer rewrite the query to make it more efficient 

Physical Plan - Spark decides how to execute the query , such as Brodcast hash join,sort merge join 

DAG - The execution Plan is representes as Directed acylic graph of operations

Stages - Spark Splits the DAG at shuffle bounderies 

Tasks - Each stage is divided into tasks 

Excutors - Actually run those Task on data 

In [ ]:
df.select("customer_id1").show()

## Catlyst Optimizer 

- Catalyst optimizer is spark SQl query optimization framework. 

- It is responsibale for analyzing and optimizing 

- Catalysyt optimizer is the intelligence inside Spark SQL that converts out Dataframe/SQl query into an optimized execution plan before spark runs the job. 




In [ ]:
result=(
    df
    .select(
        "emp_id",
        "name",
        "deparment",
        "sal",
        "city"
    )
    .filter(df.sal>50000)
    .select(
        "emp_id",
        "sal",)
)

User Query --> Parsed Logical Plan ----> Analyzer -----> Analyzed Logical Plan ----> catalyst optimizer ----> Optimized Logical Plan ------> Physical Planner ----> Physical Plan ----> Execution 

In [ ]:
df.printSchema()

rdd=parallize([
    (1,"A",50000),
    (1,"A",50000),
    (1,"A",50000),
])

result=rdd.filter(lambda x:x[2]>50000)

In [ ]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("CatalystOptimization")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled","false")
    .getOrCreate()
)
sc=spark.sparkContext

# AQE (Adaptive Query Execution) is a feature in Apache Spark that optimizes query execution plans based on runtime statistics. 
# It allows Spark to dynamically adjust the execution plan during query execution, leading to improved performance and resource utilization.

In [ ]:
print("Spark Version:", spark.version)
print("Application Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default Parallelism:", spark.sparkContext.defaultParallelism)

In [ ]:
data=[
    (1,"A", "Engineering", 60000, "New York"),
    (2,"B", "Marketing", 45000, "Los Angeles"),
    (3,"C", "Sales", 55000, "Chicago"), 
    (4,"D", "Engineering", 70000, "New York"),
    (5,"E", "Marketing", 40000, "Los Angeles"),
    (6,"F", "Sales", 60000, "Chicago"), 
    (7,"G", "Engineering", 80000, "New York"),
    (8,"H", "Marketing", 50000, "Los Angeles"),
    (9,"I", "Sales", 65000, "Chicago"),
    (10,"J", "Engineering", 75000, "New York"),
    (11,"K", "Marketing", 55000, "Los Angeles"),
    (12,"L", "Sales", 70000, "Chicago"),
     
]


columns=["emp_id", "name", "department", "sal", "city"]

df=spark.createDataFrame(data,columns)
df.show()

In [ ]:
result=(
    df.filter(df.sal > 50000)
    .select(
        "name",
        "department",
        "sal"
    )
)

In [ ]:
result.explain(True)

In [ ]:
result.show()

## catalyst Optimizer Rules:

1 Predicate Pushdoen 

    - A predicate is basically a filter consition 

    - Move the filter as close as possbile to the data sources 

    

## Detect the longest consective transection streak per customer 

transections=[
    ("C101","2026-09-01",100),
    ("C101","2026-09-02",200),
    ("C101","2026-09-04",150),
    ("C101","2026-09-05",300),
    ("C101","2026-09-06",250),

    ("C102","2026-09-01",400),
    ("C102","2026-09-03",500),
    ("C102","2026-09-04",200),
    ("C102","2026-09-05",100),
    ("C102","2026-09-06",700),

    ("C103","2026-09-02",150),
    ("C103","2026-09-03",250),
    ("C103","2026-09-04",350),
    ("C103","2026-09-05",450)

]

For Every customer, Find the longest streak of consective calender days on which the customer made t least one transection 

- Your Output should contain 
    customer_id
    streak_start_date,
    streak_end_date,
    streak_lenght,
    total_anount_during_streak 

Also handle the condition :

    ("C101","2026-09-05",300),
    ("C101","2026-09-05",400),

    # Same customer can have multiple transection on the same day 

    - Those should count as one streak day, but both amounts must contribute to the total amount 

    - If two streaks have the same maximum lenght choose the streak having the larger total transcrtion amount . If both 
      lenght and amount are equal, choose the earlier streak 




In [ ]:
import random 
members=["Vamsi","vijay","darsan","ashad","Harini","divakarn","dibyajyoti","XXXXXX"]

random.shuffle(members)
group=[]
for i in range(0,len(members),2):
    group.append(members[i:i+2])

for i,group in enumerate(group,start=1):
    print(f"Group {i}: {group}")


In [ ]:
transections=[
    ("C101","2026-09-01",100),
    ("C101","2026-09-02",200),
    ("C101","2026-09-04",150),
    ("C101","2026-09-05",300),
    ("C101","2026-09-06",250),

    ("C102","2026-09-01",400),
    ("C102","2026-09-03",500),
    ("C102","2026-09-04",200),
    ("C102","2026-09-05",100),
    ("C102","2026-09-06",700),

    ("C103","2026-09-02",150),
    ("C103","2026-09-03",250),
    ("C103","2026-09-04",350),
    ("C103","2026-09-05",450)

]

In [ ]:
from collections import defaultdict
from datetime import datetime, dattime,timedelta 

def find_longest_streak(transections):
    # Step 1: Group customer and dare 
    customer_data=defaultdict(lambda:defaultdict(int))
    for customer_id,date_str,amount in transections:
        date_obj=datetime.strptime(date_str,"%Y-%m-%d").date()
        customer_data[customer_id][date_obj]+=1
    result=[]

    # Step 2 : Process Rach customer 

    for customer_id,date_amounts in customer_data.items():
        best_start=None 
        best_end=None
        Best_lenght=0
        best_amount=0

        dates=date_amounts.keys()

        for current_date in dates:
            previous_date=current_date-timedelta(days=1)
            # If previous day does not exist 
            # THis is begining of a new streak
            if previous_date not in dates:
                streak_start=current_date
                streak_end=current_date
                streak_length=1
                streak_amount=date_amounts[current_date]

                # Extend the streak 
                # Keep moving forward while next daay is exist 
                while True:
                    next_date=streak_end+timedelta(days=1)
                    if next_date in dates:
                        streak_end=next_date
                        streak_length+=1
                        streak_amount+=date_amounts[next_date]
                    else:
                        break

                # Update best streak if needed 
                # Comapre current streak with bexr streak
                if (streak_length>Best_lenght) or (streak_length==Best_lenght and streak_amount>best_amount):
                    best_start=streak_start
                    best_end=streak_end
                    Best_lenght=streak_length
                    best_amount=streak_amount
            

1. Predicate Pushdown 

    - Predicate - Filter Condition 
    - Move the filter condition as close as possible to the data source 
    - Benfits 
        - Lest Disk I/O 
        - Less CPU 
        - Less Memory pressure 
        - Less Processing TIme 

    - Catalyst doesn't physically read parquet itself 

2. Column pruning 

    - Column purining means Spark tries to read only the columns actually required by the query 
    - Column pruning - Remove unnecessary columns as ealry as possible 

    Note: Predicate Pushdown - Reduces rows 
          Column pruning - Reduces the columns

3. Constant Folding 

    - Constant folding means spark evalutes expression made only from contant values during query optimization 
    
4. Boolean Simplification 

    - Reduces unnecessarily complex boolean conditions sipler ones 

    - Boolean simplification is a catalyst optmization that reduces unnecessary boolean expression. 
     
      for Example:

       - Condtion and true 

        ----> Can simplified to :

            and: 

            condtion and false can also be simplified 

            condition 

     - It oftern works togather with constant folding becasue conatnt expression may fiest become true or false  and then boolean Simplification removes the unnecessary boolean logic 


     
    
5. Null Propagation 

 - Null propagation means catalyst simplifies expression when part of expression is known to be NULL 

 - 
6. Combine filter 
    - MERGE MULTIPLE FILTERS INTO ONE COMBINED  CONDITION 

7. Partition Pruning - Avoid scanning partitions that cannot satisfy the filter 

8. Join Optimization 

    - 
9. Cost Based Optimization  # -------# ---------# 

In [ ]:
from pyspark.sql import functions as F 
df=(
    spark.range(1,100001)
    .withColumn(
        "department",
        F.when(F.col("id")%4==0,"IT")
        .when(F.col("id")%4==1,"HR")
        .when(F.col("id")%4==2,"Finance")
        .otherwise("Sales")
    )
    .withColumn(
        "salary",
        (F.col("id")%100000)+30000
    )
    .withColumn(
        "country",
        F.when(F.col("id")%3==0,"India")
        .when(F.col("id")%3==1,"USA")
        .otherwise("UK")
    )
)

In [ ]:
df.show(10)

In [ ]:
df.write.mode("overwrite").parquet("employee_data.parquet")

In [ ]:
parquet_df=spark.read.parquet("employee_data.parquet")

In [ ]:
parquet_df.show(30)

In [ ]:
parquet_df.explain("formatted")

In [ ]:
from pyspark.sql import functions as F 
filtered_df=(
    parquet_df.filter(F.col("country")=="India")
)
filtered_df.explain("formatted")

In [ ]:
result=parquet_df.select(
    "id",
    "salary"
)

result.explain("formatted")

In [ ]:
from pyspark.sql import functions as F 
constant_df=parquet_df.select(
    "id",
    (F.lit(10)+F.lit(20)).alias("constant_value")
)
constant_df.explain(True)

In [ ]:
df.filter(
    (F.col("salary")>70000) & F.lit(True)
)

In [ ]:
from pyspark.sql import functions as F 
null_df=parquet_df.select(
    "id",
    F.lit(None).cast("int")+F.lit(10).alias("null_plus_10")
)
null_df.explain(True)

In [ ]:
result=(
    parquet_df
    .filter(F.col("country")=="India")
    .filter(F.col("salary")>70000)
)
result.explain(True)

In [ ]:
result=(
    parquet_df
    .filter((F.col("salary")+1000)>70000)
    .filter(F.col("country")=="India")
)

In [ ]:
parquet_df.show(5)

In [ ]:
result=parquet_df.filter(
    F.col("salary")>70000
).select(
    "department",
    "salary"
)


In [ ]:
result=(
    parquet_df
    .groupBy("department")
    .agg(
        F.avg("salary").alias("avg_salary"),
    )
    .filter(
        F.col("avg_salary")>70000
    )
)

In [ ]:
result.show(10)

In [ ]:
partition_df=(
    spark.range(1,100001)
    .withColumn("year",F.when(F.col("id")%2==0,2025).otherwise(2026))
    .withColumn("month",F.col("id")%3+1)
    .withColumn("amount",F.col("id")*10)
)

In [ ]:
partition_df.write.mode("overwrite").partitionBy("year","month").parquet("partitioned_data.parquet")

In [ ]:
read_part_df=spark.read.parquet("partitioned_data.parquet")

In [ ]:
read_part_df.printSchema()

In [ ]:
result=read_part_df.filter(
    (F.col("year")==2026)
)

result.explain("formatted")

In [ ]:
result2=read_part_df.filter(
    (F.col("year")==2026) & (F.col("month")==2)
)

result2.explain(True)

In [ ]:
employees=(
    spark.range(1,100001)
    .withColumn(
        "department_id",
        (F.col("id")%5)+1
    )
    .withColumn(
        "salary",
        (F.col("id")%100000)+30000
    )
)

departments=spark.createDataFrame(
    [
        (1,"IT"),
        (2,"HR"),
        (3,"Finance"),
        (4,"Sales"),
        (5,"Marketing")
    ],
    ["department_id","department_name"]
)

In [ ]:
joined_df=employees.join(
    departments,
    "department_id"
)
    

In [ ]:
joined_df.explain("formatted")

## Tungsten Engine 

    - Memoery Management 

    - Off Heap/ Unsafe Memory operations 

    - Cache Awate Compuatation 

    - Whole Stage code Genaration 

        - Genarate Java code 
        - Combine Operators 
        - Reduce Function vall overhead 

    

In [ ]:
data=[
    (1,"A",50000),
    (2,"B",60000),
    (3,"C",70000)
]
df=spark.createDataFrame(data,["id","name","salary"])


In [ ]:
row=df.first()

In [ ]:
print(row)

In [ ]:
print(type(row))

In [ ]:
jdf=df._jdf
internal_rdd=jdf.queryExecution().toRdd()
first_internal_row=internal_rdd.first()
print(first_internal_row)
print(first_internal_row.getClass().getName())

├── 2. Creating DataFrames
│   ├── From Python list - A pyspark Dataframe can be created directly from python List 
│   ├── From RDD
│   ├── Using schema
│   ├── StructType / StructField
│   └── Reading CSV / JSON / Parquet




In [1]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("DataFrameLearning")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/anujshahdeo/Documents/Full_Stack_AWS_Data_Engineering/full-stack-aws-data-engineering/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/14 20:13:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [17]:
data=[
    (1,"A",50000),
    (2,"B",60000),
    (3,"C",70000),
    (4,"D",80000)
]

In [18]:
df=spark.createDataFrame(
    data,["id","name","salary"]
)

# createDataframe(data,schema)

In [19]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)



In [ ]:
d

In [12]:
df2=spark.createDataFrame(
    data
)
df2.show()

+---+---+-----+
| _1| _2|   _3|
+---+---+-----+
|  1|  A|50000|
|  2|  B|60000|
|  3|  C|70000|
|  4|  D|80000|
+---+---+-----+



CreateDataframe() : 

Data- Actual data that you want to convert into a spark Dataframe 

Spark can accept diffrent input type:

    - RDD
    - Python List 
    - tuple based data 
    - Row Objects 
    - dictionary data 
    - pandas Dataframe 
    - NumPy Array 
    - PyArrow Tables 

    

spark.creatDataFrame(
    data,
    schema=None
    samplingRatio=None,
    verifySchema=True
)

Case 1 - Schama=None 

Schama=None 

- This is a default 
df=spark.createDataframe(data) - 

Case -2 : Schema as a list of column names 

df=spark.createDataFrame(
    data,["id","name","salary"]
)

Case-3 Explicit StructType Schema 

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

schema=StructType([
    StructField("id",IntegerType(),False),
    StructField("name",StringType(),True)
])

data=[
    (1,"A"),
    (2,"B")
]

df=spark.createDataFrame(data,schema)
df.show()
df.printSchema()


case -4 Schema as DDL String 

df=spark.createdataFrame(
    data,"id int,name string"
)




In [21]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

schema=StructType([
    StructField("id",IntegerType(),False),
    StructField("name",StringType(),True)
])

data=[
    (1,"A"),
    (2,"B")
]

df=spark.createDataFrame(data,schema)
df.show()
df.printSchema()

+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
+---+----+

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = true)



In [13]:
from pyspark.sql import Row
data=[
    Row(id=1,name="A"),
    Row(id=2,name="B")
]
df=spark.createDataFrame(data)

In [14]:
df.show()

+---+----+
| id|name|
+---+----+
|  1|   A|
|  2|   B|
+---+----+



In [15]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [5]:
df.show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  1|   A| 50000|
|  2|   B| 60000|
|  3|   C| 70000|
|  4|   D| 80000|
+---+----+------+



In [6]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)



In [7]:
rdd=spark.sparkContext.parallelize(data)

In [8]:
rdd.collect()

[(1, 'A', 50000), (2, 'B', 60000), (3, 'C', 70000), (4, 'D', 80000)]

In [10]:
df1=spark.createDataFrame(
    rdd,["id","name","salary"]
)
df1.show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  1|   A| 50000|
|  2|   B| 60000|
|  3|   C| 70000|
|  4|   D| 80000|
+---+----+------+



In [22]:
rdd=spark.sparkContext.parallelize([
    (1,"A"),
    (2,"B"),
    (3,"C"),
    (4,"D")
])



In [23]:
df=spark.createDataFrame(
    rdd,
    schema=None,
    samplingRatio=0.5
)

In [24]:
df.show()

+---+---+
| _1| _2|
+---+---+
|  1|  A|
|  2|  B|
|  3|  C|
|  4|  D|
+---+---+



In [25]:
df.printSchema()

root
 |-- _1: long (nullable = true)
 |-- _2: string (nullable = true)



In [30]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

schema=StructType([
    StructField("id",IntegerType(),False),
    StructField("name",StringType(),True)
])

data=[
    (1,"A"),
    ("One","B"),
    (2,"C"),
    (3,"D")
]

df=spark.createDataFrame(data,schema,verifySchema=True)
df.show()
df.printSchema()

PySparkTypeError: [FIELD_DATA_TYPE_UNACCEPTABLE_WITH_NAME] field id: IntegerType() can not accept object 'One' in type <class 'str'>.

## Explicit Schema 
    - StructType 
    - StructFeild
    - Column Name
    - Datatype
    - Nullable
    - Metadata

- What is an explicit Schema?

    - An explicit schema means we tell spark exactly how the datframe should look intead of asking spark to infer it. 

    
StructType - Entire table structure 
    |
    |
    ------StructField - Definition of one column 
                |
                |
                --id 
                - IntergerType
                - True 

    -------StructField 
                |
                |
                -- name 
                -- StringType 
                -- true 

    ------- StructField
                |
                |
                -- salary
                -- DoubleType
                -- True 



In [ ]:
# Explicit Schema 

schema=StructType([
    StructField("id",IntegerType(),False),
    StructField("Name",StringType(),False),
    StructField("Salary",DoubleType(),True),
    StructField("id",Integerpe(),False),
    StructField("id",IntegerType(),False),
    StructField("id",IntegerType(),False),
    StructField("id",IntegerType(),False),
    StructField("id",IntegerType(),False),
    StructField("id",IntegerType(),False)
])

In [38]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

schema=StructType([
    StructField("emp_id",IntegerType(),True), 
    StructField("emp_name",IntegerType(),True),
    StructField("department",StringType(),True),
    StructField("salary",IntegerType(),True)         
])
df=spark.read.csv(
    "employee.txt",
    schema=schema,
    header=False)

df.show()

df.printSchema()

+------+--------+----------+------+
|emp_id|emp_name|department|salary|
+------+--------+----------+------+
|   101|    NULL|        IT| 50000|
|   102|    NULL|        HR| 30000|
|   103|    NULL|       FIN| 70000|
|   104|    NULL|        HR| 60000|
|   105|    NULL|        IT| 90000|
+------+--------+----------+------+

root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)



In [39]:
schema="emp_id int,emp_name string,department string,salary int"
df=spark.read.csv(
    "employee.txt",
    schema=schema,
    header=False
)
df.show()
df.printSchema()


+------+--------+----------+------+
|emp_id|emp_name|department|salary|
+------+--------+----------+------+
|   101|    John|        IT| 50000|
|   102|      Jo|        HR| 30000|
|   103|     ohn|       FIN| 70000|
|   104|      on|        HR| 60000|
|   105|      Jh|        IT| 90000|
+------+--------+----------+------+

root
 |-- emp_id: integer (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)



In [36]:
df.show()

+---+----+---+-----+
|_c0| _c1|_c2|  _c3|
+---+----+---+-----+
|101|John| IT|50000|
|102|  Jo| HR|30000|
|103| ohn|FIN|70000|
|104|  on| HR|60000|
|105|  Jh| IT|90000|
+---+----+---+-----+

